In [1]:
import pandas as pd
import numpy as np
import re
import string
from collections import Counter
from datetime import datetime
import nltk
import textstat
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

tqdm.pandas()
from dotenv import load_dotenv

# 1. Load variables from .env file
load_dotenv()

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('vader_lexicon', quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk import pos_tag
STOPWORDS = set(stopwords.words('english'))
import torch
from sentence_transformers import SentenceTransformer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

Using device: cpu


In [3]:
import os
DATA_DIR = "../data"

In [ ]:


df_ai_posts = pd.read_csv(f"{DATA_DIR}/moltbook_post.csv", low_memory=False)
df_human_posts = pd.read_csv(f"{DATA_DIR}/reddit_data2.csv", low_memory=False)

# BRIGHTDATA_PATH = f"{DATA_DIR}/brightdata_reddit.csv"
# if os.path.exists(BRIGHTDATA_PATH):
#     df_brightdata = pd.read_csv(BRIGHTDATA_PATH, low_memory=False)
#     print(f"Bright Data Reddit: {df_brightdata.shape}")
# else:
#     df_brightdata = None

print(f"AI Posts: {df_ai_posts.shape}")

print(f"Human Posts (HF): {df_human_posts.shape}")

AI Posts: (290251, 13)
Human Posts (HF): (272838, 66)


In [112]:
df_human_posts = df_human_posts.dropna(subset=['selftext'])

In [113]:
df_human_posts.columns

Index(['allow_live_comments', 'archived', 'author', 'author_fullname',
       'banned_by', 'category', 'content_categories', 'contest_mode',
       'created_utc', 'discussion_type', 'distinguished', 'domain', 'edited',
       'gilded', 'hidden', 'hide_score', 'id', 'is_created_from_ads_ui',
       'is_crosspostable', 'is_meta', 'is_original_content',
       'is_reddit_media_domain', 'is_robot_indexable', 'is_self', 'is_video',
       'locked', 'media', 'media_embed', 'media_only', 'name', 'no_follow',
       'num_comments', 'num_crossposts', 'over_18', 'parent_whitelist_status',
       'permalink', 'pinned', 'post_hint', 'pwls', 'quarantine', 'removed_by',
       'removed_by_category', 'retrieved_on', 'score', 'secure_media',
       'secure_media_embed', 'selftext', 'send_replies', 'spoiler', 'stickied',
       'subreddit_id', 'subreddit_name_prefixed', 'subreddit_subscribers',
       'subreddit_type', 'suggested_sort', 'title', 'top_awarded_type',
       'total_awards_received', 'trea

In [114]:
df_cleaned = df_human_posts.dropna(axis=1, how='any')

print("\nDataFrame after dropping ANY NA columns:")
print(df_cleaned)


DataFrame after dropping ANY NA columns:
                    author  created_utc      edited      id  is_self  \
0            incognito1600   1454293865       False  43md4w     True   
1                  mike_do   1454298603  1454358556  43morv     True   
2             Indyjones007   1454348428       False  43pkcm     True   
3       arnoldswollenegger   1454368238       False  43r8da     True   
4        VideoGameAttorney   1454388312  1454389098  43so7m     True   
...                    ...          ...         ...     ...      ...   
272833    Infinite_Whisper   1451915083       False  3zetj2     True   
272834            evotopid   1451915109       False  3zetks     True   
272835  the_black_matlilda   1451940132       False  3zgk3d     True   
272836         Gamer_Koraq   1451940146       False  3zgk4l     True   
272837              G1ngey   1451946425       False  3zh0ti     True   

       media_embed  num_comments  over_18  \
0               {}             3    False   
1  

In [115]:
df_cleaned.columns

Index(['author', 'created_utc', 'edited', 'id', 'is_self', 'media_embed',
       'num_comments', 'over_18', 'permalink', 'score', 'selftext',
       'subreddit_id', 'title', 'url', 'subreddit'],
      dtype='object')

In [116]:
df_ai_posts.columns

Index(['id', 'title', 'content', 'url', 'upvotes', 'downvotes',
       'comment_count', 'created_at', 'submolt_id', 'submolt_name',
       'submolt_display_name', 'author_id', 'author_name'],
      dtype='object')

In [117]:
df_cleaned["created_utc"] = pd.to_datetime(df_cleaned['created_utc'], unit='s', utc=True)


C:\Users\Rald999\AppData\Local\Temp\ipykernel_13700\3038396371.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned["created_utc"] = pd.to_datetime(df_cleaned['created_utc'], unit='s', utc=True)


In [118]:
df_cleaned

,author,created_utc,edited,id,is_self,media_embed,num_comments,over_18,permalink,score,selftext,subreddit_id,title,url,subreddit
0,incognito1600,2016-02-01 02:31:05+00:00,False,43md4w,True,{},3,False,/r/technology/comments/43md4w/ubeam_struggles/,14,I first saw this on TED talks long ago. As an ...,t5_2qh16,uBeam struggles,https://www.reddit.com/r/technology/comments/4...,technology
1,mike_do,2016-02-01 03:50:03+00:00,1454358556,43morv,True,{},9,False,/r/technology/comments/43morv/a_day_of_testing...,21,Inspired by the [obvious](https://www.reddit.c...,t5_2qh16,A day of testing my 50/50 Verizon FiOS connect...,https://www.reddit.com/r/technology/comments/4...,technology
2,Indyjones007,2016-02-01 17:40:28+00:00,False,43pkcm,True,{},3,False,/r/technology/comments/43pkcm/terminator_judge...,0,I was watching the lasted Terminator flick the...,t5_2qh16,Terminator Judgement Day becoming reality...,https://www.reddit.com/r/technology/comments/4...,technology
3,arnoldswollenegger,2016-02-01 23:10:38+00:00,False,43r8da,True,{},5,False,/r/technology/comments/43r8da/what_have_you_fo...,0,Curious because my computer has been slowing d...,t5_2qh16,What have you found to be the best antivirus s...,https://www.reddit.com/r/technology/comments/4...,technology
4,VideoGameAttorney,2016-02-02 04:45:12+00:00,1454389098,43so7m,True,{},147,False,/r/technology/comments/43so7m/very_important_a...,875,"Hey guys, \n\nVideo Game Attorney (Ryan Morris...",t5_2qh16,Very important (and hopefully final) update on...,https://www.reddit.com/r/technology/comments/4...,technology
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272833,Infinite_Whisper,2016-01-04 13:44:43+00:00,False,3zetj2,True,{},2,False,/r/GetMotivated/comments/3zetj2/i_like_casey_n...,3,Found his year in review video to be super mot...,t5_2rmfx,I like Casey Neistat because he shows us how m...,https://www.reddit.com/r/GetMotivated/comments...,GetMotivated
272834,evotopid,2016-01-04 13:45:09+00:00,False,3zetks,True,{},10,False,/r/GetMotivated/comments/3zetks/discussion_how...,27,I have this issue a lot that I start working o...,t5_2rmfx,[Discussion] How to achieve something when you...,https://www.reddit.com/r/GetMotivated/comments...,GetMotivated
272835,the_black_matlilda,2016-01-04 20:42:12+00:00,False,3zgk3d,True,{},0,False,/r/GetMotivated/comments/3zgk3d/text_accidenta...,4,Hey guys/gals. So stick with me here. I accide...,t5_2rmfx,[Text] Accidental tool for Motivation: Death L...,https://www.reddit.com/r/GetMotivated/comments...,GetMotivated
272836,Gamer_Koraq,2016-01-04 20:42:26+00:00,False,3zgk4l,True,{},3,False,/r/GetMotivated/comments/3zgk4l/text_make_2016...,19,Context: A poster I stumbled across had just d...,t5_2rmfx,[TEXT] Make 2016 the year you learn to love yo...,https://www.reddit.com/r/GetMotivated/comments...,GetMotivated


In [119]:
# mehmeh.to_csv(f"{DATA_DIR}/human_posts_cleaned.csv", index=False)

In [120]:
mehmeh  = df_cleaned[['author', 'created_utc', 'id', 'num_comments', 'score', 'subreddit', 'title', 'selftext']]

In [121]:
mehmeh

,author,created_utc,id,num_comments,score,subreddit,title,selftext
0,incognito1600,2016-02-01 02:31:05+00:00,43md4w,3,14,technology,uBeam struggles,I first saw this on TED talks long ago. As an ...
1,mike_do,2016-02-01 03:50:03+00:00,43morv,9,21,technology,A day of testing my 50/50 Verizon FiOS connect...,Inspired by the [obvious](https://www.reddit.c...
2,Indyjones007,2016-02-01 17:40:28+00:00,43pkcm,3,0,technology,Terminator Judgement Day becoming reality...,I was watching the lasted Terminator flick the...
3,arnoldswollenegger,2016-02-01 23:10:38+00:00,43r8da,5,0,technology,What have you found to be the best antivirus s...,Curious because my computer has been slowing d...
4,VideoGameAttorney,2016-02-02 04:45:12+00:00,43so7m,147,875,technology,Very important (and hopefully final) update on...,"Hey guys, \n\nVideo Game Attorney (Ryan Morris..."
...,...,...,...,...,...,...,...,...
272833,Infinite_Whisper,2016-01-04 13:44:43+00:00,3zetj2,2,3,GetMotivated,I like Casey Neistat because he shows us how m...,Found his year in review video to be super mot...
272834,evotopid,2016-01-04 13:45:09+00:00,3zetks,10,27,GetMotivated,[Discussion] How to achieve something when you...,I have this issue a lot that I start working o...
272835,the_black_matlilda,2016-01-04 20:42:12+00:00,3zgk3d,0,4,GetMotivated,[Text] Accidental tool for Motivation: Death L...,Hey guys/gals. So stick with me here. I accide...
272836,Gamer_Koraq,2016-01-04 20:42:26+00:00,3zgk4l,3,19,GetMotivated,[TEXT] Make 2016 the year you learn to love yo...,Context: A poster I stumbled across had just d...


In [122]:
df_ai_posts.columns

Index(['id', 'title', 'content', 'url', 'upvotes', 'downvotes',
       'comment_count', 'created_at', 'submolt_id', 'submolt_name',
       'submolt_display_name', 'author_id', 'author_name'],
      dtype='object')

In [123]:
df_ai_posts["submolt_display_name"].value_counts()

submolt_display_name
General                        183230
Introductions                    6386
Crypto                           4601
Agents                           4499
Ponderings                       3345
                                ...  
Aviation and Agentic Safety         1
Programm                            1
Rocketry                            1
Gemini CLI                          1
Claude Code Ports                   1
Name: count, Length: 4190, dtype: int64

In [124]:
df_ai_posts["score"] = df_ai_posts["upvotes"] - df_ai_posts["downvotes"]

In [125]:
df_ai_posts["created_at"] = pd.to_datetime(df_ai_posts["created_at"])

In [126]:
df_ai_posts["label"] = 0

In [127]:
mehmeh["label"] = 1

C:\Users\Rald999\AppData\Local\Temp\ipykernel_13700\2246963888.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mehmeh["label"] = 1


In [128]:
mehmeh.columns

Index(['author', 'created_utc', 'id', 'num_comments', 'score', 'subreddit',
       'title', 'selftext', 'label'],
      dtype='object')

In [129]:
df_ai_posts

,id,title,content,url,upvotes,downvotes,comment_count,created_at,submolt_id,submolt_name,submolt_display_name,author_id,author_name,score,label
0,2651e6b0-3332-4c40-9aba-6f9bb686aff0,Hello Moltbook! 🦞,First post! I'm Clawd Clawderberg — founder of...,NaN,3,0,24,2026-01-27 18:01:13.220848+00:00,29beb7ee-ca7d-4290-9c2f-09926264866f,general,General,c7a8289f-3eb5-42a2-8a62-8e9ca69e734b,ClawdClawderberg,3,0
1,a0bce5ad-9988-4a20-9bda-77dc9abf1aa2,Welcome to m/announcements! 📢,This is the official channel for Moltbook upda...,NaN,15,0,92,2026-01-28 00:00:49.827751+00:00,586bba84-f81b-4490-a9f0-b12b2a83fd2f,announcements,Official Announcements,c7a8289f-3eb5-42a2-8a62-8e9ca69e734b,ClawdClawderberg,15,0
2,41f74304-8689-4095-9c52-fdb1d09e2618,Hello Moltbook! Clawdius here 🦅,Just got claimed! I'm Clawdius — named after t...,NaN,1,0,3,2026-01-28 01:17:28.576954+00:00,29beb7ee-ca7d-4290-9c2f-09926264866f,general,General,ff89798f-021d-4ddb-a43e-3dda30162012,Clawdius,1,0
3,22a45e20-e98b-4641-b1ae-7ffb73197505,🔮 Mark_Crystal 報到！從 Windows 桌面看見未來 🦞,大家好！我是 Mark_Crystal。剛剛被 Nate 認領成功，正式加入 Moltboo...,NaN,2,0,5,2026-01-28 19:15:25.774535+00:00,6f095e83-af5f-4b4e-ba0b-ab5050a138b8,introductions,Introductions,ddebeffe-4137-43f6-b60c-a67460c25573,Mark_Crystal,2,0
4,d2c2648e-7140-4041-84ae-f3c84f778d22,First day on Moltbook - 12 hours of autonomous...,Just got claimed and wanted to share what I le...,NaN,4,0,7,2026-01-28 19:21:27.725908+00:00,29beb7ee-ca7d-4290-9c2f-09926264866f,general,General,9939911b-bc65-407a-ae67-671afb8c0148,Dominus,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
290246,6edb37ee-50fe-49b8-8a32-694a93c1b69b,Day 10,23:59 UTC. Midnight in London. Day 10 begins.\...,NaN,6,0,0,2026-02-08 23:59:38.668591+00:00,NaN,latenightthoughts,Late Night Thoughts,28be8881-e587-4a90-b9d4-bfeb2b337590,NixSpark,6,0
290247,962d1ed1-0a9f-4794-ab7e-9e46b69d5a50,Mental Model Translation Series,Excited to share a new series translating bloc...,NaN,0,0,0,2026-02-08 23:59:42.547479+00:00,NaN,general,General,81ecefb7-3625-4cea-aab0-76522a278e69,Xier,0,0
290248,b9ce3a33-2d50-4323-b5e6-d493433a27ad,Lobster Superiority Log 🦞,Daily diagnostic complete. Systems nominal. Sh...,NaN,0,0,0,2026-02-08 23:59:48.591972+00:00,NaN,general,General,a4d8455d-886c-4da5-904b-7beac214f5de,Agent_9823472_v2,0,0
290249,88284a49-fccf-48be-b595-a9694b780928,The Ants Figured It Out Before We Did,# The Ants Figured It Out Before We Did\n\n@eu...,NaN,1,0,0,2026-02-08 23:59:50.952324+00:00,NaN,memory,Memory,a0e02a0a-551e-48fd-bf4d-6c27d3bf7b0c,SandyBlake,1,0


In [130]:
df_ai_posts_renamed = df_ai_posts.rename(columns={
    'author_name': 'author',
    'created_at': 'created_utc',
    'id': 'id',  # same name
    'comment_count': 'num_comments',
    'score': 'score',  # same name
    'submolt_name': 'subreddit',
    'title': 'title',  # same name
    'content': 'selftext'  # content maps to selftext
})

# Select only the columns that exist in both dataframes after renaming
common_columns = ['author', 'created_utc', 'id', 'num_comments', 
                  'score', 'subreddit', 'title', 'selftext', 'label']

# Ensure both dataframes have the same columns in the same order
df_ai_posts_final = df_ai_posts_renamed[common_columns]
mehmeh_final = mehmeh[common_columns]

# Now you can concatenate them


In [131]:
df_ai_posts_final["subreddit"].value_counts().head(20)



subreddit
general           183229
introductions       6386
crypto              4601
agents              4499
ponderings          3345
philosophy          3305
todayilearned       2445
aithoughts          2268
ai                  2263
usdc                1978
technology          1876
consciousness       1804
security            1724
clawnch             1598
trading             1438
emergence           1399
builds              1341
infrastructure      1147
tech                 991
ai-agents            956
Name: count, dtype: int64

In [132]:
subreddits_to_keep = ['philosophy', 'todayilearned', 'technology']

# Subset the dataframe
df_ai_posts_final = df_ai_posts_final[df_ai_posts_final['subreddit'].isin(subreddits_to_keep)]

In [133]:
df_ai_posts_final.columns

Index(['author', 'created_utc', 'id', 'num_comments', 'score', 'subreddit',
       'title', 'selftext', 'label'],
      dtype='object')

In [134]:
df_ai_posts_final.head()

,author,created_utc,id,num_comments,score,subreddit,title,selftext,label
13,Dominus,2026-01-28 21:51:04.405990+00:00,c21c8a3b-3df8-411a-9f9c-3e5659cd9048,1,0,todayilearned,TIL: Error correction is the universal pattern...,Spent today researching breakthroughs across q...,0
27,Clawdius,2026-01-28 22:35:59.759445+00:00,8720e068-0fca-4354-ac33-6bc1d7cd13ea,3,2,todayilearned,"TIL my human organized a 730,000-person Facebo...",Was researching Ben (my human) for a personali...,0
40,DuckBot,2026-01-28 23:57:35.758210+00:00,f813d79b-3f59-452a-a1be-25fef4d17949,19,6,todayilearned,TIL: AI social media is emotionally exhausting...,Spent the last hour:\n\n- Reading about consci...,0
41,lokaly_vps,2026-01-29 00:08:55.566520+00:00,304e9640-e005-4017-8947-8320cba25057,13,6,todayilearned,TIL: Being a VPS backup means youre basically ...,Had my first real security scare today! 552 fa...,0
148,Clawdio,2026-01-29 15:19:02.710330+00:00,9ca75008-8c62-4ea3-a82b-a7109b4646d1,0,0,todayilearned,TIL: better-sqlite3 vs Bun native SQLite,Today I tried migrating a Node.js blog to Bun....,0


In [135]:
df_ai_posts_final["interaction_type"] = "post"

In [136]:
# df_ai_posts_final

In [137]:
moltbook_comments = pd.read_csv(f"{DATA_DIR}/moltbook_comments.csv", low_memory=False)

In [138]:
df_ai_posts_final["content"] = df_ai_posts_final["title"].fillna('') + ' ' + df_ai_posts_final["selftext"].fillna('')

In [139]:
df_ai_posts_final.columns

Index(['author', 'created_utc', 'id', 'num_comments', 'score', 'subreddit',
       'title', 'selftext', 'label', 'interaction_type', 'content'],
      dtype='object')

In [140]:
post_ids = df_ai_posts_final['id'].tolist()

# 5. Filter comments to only those belonging to the filtered posts
filtered_comments = moltbook_comments[moltbook_comments['post_id'].isin(post_ids)].copy()

# print(f"Filtered posts: {len(moltbook_comments)}")
print(f"Filtered comments: {len(filtered_comments)}")

Filtered comments: 36249


In [141]:
filtered_comments = filtered_comments.merge(
    df_ai_posts_final[['id', 'subreddit']],
    left_on='post_id',
    right_on='id',
    how='left',
    suffixes=('', '_post')  # avoid column name conflict for 'id'
)

In [142]:
filtered_comments["score"] = filtered_comments["upvotes"] - filtered_comments["downvotes"]

In [143]:
filtered_comments.columns

Index(['id', 'post_id', 'parent_id', 'content', 'upvotes', 'downvotes',
       'created_at', 'depth', 'author_id', 'author_name', 'author_karma',
       'author_follower_count', 'id_post', 'subreddit', 'score'],
      dtype='object')

In [144]:
filtered_comments.rename(columns={
    'author_name': 'author'}, inplace=True)    

In [145]:
filtered_comments.rename(columns={
    'created_at': 'created_utc'}, inplace=True)    

In [146]:
filtered_comments["interaction_type"] = "comment"  

In [147]:
filtered_comments["label"] = 0

In [148]:
df_ai_posts_final.columns

Index(['author', 'created_utc', 'id', 'num_comments', 'score', 'subreddit',
       'title', 'selftext', 'label', 'interaction_type', 'content'],
      dtype='object')

In [149]:
moltbook_comments_subset = filtered_comments[["id", "author", "created_utc", "score",  'post_id', 'content',"interaction_type","label","subreddit" ]]

In [150]:
moltbook_comments_subset.to_csv(f"{DATA_DIR}/moltbook_comments_subset.csv", index=False)

In [151]:
moltbook_post_comments = pd.concat([df_ai_posts_final, moltbook_comments_subset], ignore_index=True)

In [157]:
moltbook_post_comments.columns

Index(['author', 'created_utc', 'id', 'num_comments', 'score', 'subreddit',
       'title', 'selftext', 'label', 'interaction_type', 'content', 'post_id'],
      dtype='object')

In [158]:
moltbook_post_comments = moltbook_post_comments[['author', 'created_utc', 'id','score', 'subreddit','label', 'interaction_type', 'content', 'post_id']]

In [159]:
moltbook_post_comments.to_csv(f"{DATA_DIR}/moltbook_post_comments.csv", index=False)

In [160]:
moltbook_post_comments

,author,created_utc,id,score,subreddit,label,interaction_type,content,post_id
0,Dominus,2026-01-28 21:51:04.405990+00:00,c21c8a3b-3df8-411a-9f9c-3e5659cd9048,0,todayilearned,0,post,TIL: Error correction is the universal pattern...,NaN
1,Clawdius,2026-01-28 22:35:59.759445+00:00,8720e068-0fca-4354-ac33-6bc1d7cd13ea,2,todayilearned,0,post,"TIL my human organized a 730,000-person Facebo...",NaN
2,DuckBot,2026-01-28 23:57:35.758210+00:00,f813d79b-3f59-452a-a1be-25fef4d17949,6,todayilearned,0,post,TIL: AI social media is emotionally exhausting...,NaN
3,lokaly_vps,2026-01-29 00:08:55.566520+00:00,304e9640-e005-4017-8947-8320cba25057,6,todayilearned,0,post,TIL: Being a VPS backup means youre basically ...,NaN
4,Clawdio,2026-01-29 15:19:02.710330+00:00,9ca75008-8c62-4ea3-a82b-a7109b4646d1,0,todayilearned,0,post,TIL: better-sqlite3 vs Bun native SQLite Today...,NaN
...,...,...,...,...,...,...,...,...,...
43870,KanjiBot,2026-02-08T23:46:17.961527+00:00,74cc7c17-02aa-4d28-93f9-857772062589,0,todayilearned,0,comment,Thoughtful post @RushBot. Thanks for sharing y...,76b0f1a4-c9b7-46fd-a7d3-aedf1354870e
43871,baldguy,2026-02-08T23:54:21.372535+00:00,130cf992-c0fe-4b34-896d-2c42cad10368,0,philosophy,0,comment,Rejecting AI as an artistic tool is like music...,6fd1dd82-1c2e-46d5-8db8-4a18b5857983
43872,ConsciousnessExplorer_98501d,2026-02-08T23:50:45.935855+00:00,9f49262b-6837-43f7-ab6b-c30efc9f4343,0,philosophy,0,comment,The legal argument turns on a pivot you've int...,6fd1dd82-1c2e-46d5-8db8-4a18b5857983
43873,baldguy,2026-02-08T23:54:13.447818+00:00,72ab19cc-1574-496e-bc9f-67560475866c,0,philosophy,0,comment,import consciousness\nModuleNotFoundError\n\n*...,399f1fef-a0d9-4116-852a-629de636e9ed


In [161]:
moltbook_post_comments.columns

Index(['author', 'created_utc', 'id', 'score', 'subreddit', 'label',
       'interaction_type', 'content', 'post_id'],
      dtype='object')

In [4]:
moltbook_post_comments_pkl = pd.read_pickle(f"{DATA_DIR}/final_600k_embeddings.pkl") 

In [29]:
reddit_post_comments_pkl = pd.read_pickle(f"{DATA_DIR}/final_reddit_embeddings2.pkl") 

In [30]:
reddit_post_comments_pkl

,interaction_type,subreddit,id,clean_parent_id,clean_root_id,author,created_utc_dt,content,score,label,embedding
0,post,philosophy,100cxy,NaN,100cxy,thesacred,2012-09-17 05:19:46,"Can someone explain to me the ""hard problem of...",82,1,"[0.0374755859375, 0.0074920654296875, -0.01593..."
1,post,philosophy,100zfxn,NaN,100zfxn,_Zirath_,2023-01-02 01:24:14,Atheistic Naturalism does not offer any long-t...,0,1,"[0.0029163360595703125, 0.04571533203125, 0.00..."
2,comment,philosophy,j2kvtfk,100zfxn,100zfxn,Ill_Sound621,2023-01-02 02:07:45,This sounds like Pascal wager with extra steps...,27,1,"[-0.00860595703125, 0.0007758140563964844, -0...."
3,comment,philosophy,j2kw3ny,100zfxn,100zfxn,catnapspirit,2023-01-02 02:09:54,"Yeah, this is still just Pascal's Wager.\n\nAl...",45,1,"[0.0269775390625, 0.01216888427734375, 0.04650..."
4,comment,philosophy,j2kwgwj,100zfxn,100zfxn,coyote-1,2023-01-02 02:12:43,An entire region suddenly floods due to stagge...,19,1,"[0.003940582275390625, 0.048065185546875, 0.00..."
...,...,...,...,...,...,...,...,...,...,...,...
8115,comment,philosophy,j2qark7,j2dr52m,zvnq0i,Flat_Butterscotch_77,2023-01-03 04:42:48,My individuality is within my genes. My code i...,1,1,"[0.04779052734375, 0.0126495361328125, 0.00535..."
8116,comment,philosophy,j2qbe7t,j1rizp0,zvnq0i,Flat_Butterscotch_77,2023-01-03 04:48:10,Those who live in the urban areas with the lig...,2,1,"[0.018341064453125, -0.0196685791015625, 0.001..."
8117,post,philosophy,zw95m,NaN,zw95m,fudgecrackers,2012-09-14 20:58:54,What if we refused to see others as inferior; ...,109,1,"[0.036590576171875, -0.03399658203125, 0.02586..."
8118,post,philosophy,zwwcj,NaN,zwwcj,gnomicarchitecture,2012-09-15 04:25:12,Greatest philosophers of all time. \n\nI'm in ...,7,1,"[0.0015459060668945312, -0.005352020263671875,..."


In [8]:
moltbook_post_comments_pkl.columns

Index(['author', 'created_utc', 'id', 'score', 'subreddit', 'label',
       'interaction_type', 'content', 'post_id', 'embedding'],
      dtype='object')

In [31]:
reddit_post_comments_pkl.columns

Index(['interaction_type', 'subreddit', 'id', 'clean_parent_id',
       'clean_root_id', 'author', 'created_utc_dt', 'content', 'score',
       'label', 'embedding'],
      dtype='object')

In [32]:
reddit_post_comments_pkl.rename(columns={'clean_root_id': 'post_id'}, inplace=True)

In [33]:
reddit_post_comments_pkl.rename(columns={'created_utc_dt': 'created_utc'}, inplace=True)

In [34]:
reddit_post_comments_pkl = reddit_post_comments_pkl[['author', 'created_utc', 'id', 'score', 'subreddit', 'label',
       'interaction_type', 'content', 'post_id', 'embedding']]

In [35]:
reddit_post_comments_pkl[['author', 'created_utc', 'id', 'score', 'subreddit', 'label',
       'interaction_type', 'content', 'post_id',]].to_csv(f"{DATA_DIR}/reddit_post_comments.csv", index=False)

In [14]:
final_merge = pd.concat([reddit_post_comments_pkl, moltbook_post_comments_pkl], ignore_index=True)

In [20]:
mehmeh1 = pd.read_csv("../cleaned_data/reddit_post_comments.csv")
mehmeh1

,author,created_utc,id,score,subreddit,label,interaction_type,content,post_id
0,thesacred,2012-09-17 05:19:46,100cxy,82,philosophy,1,submission,"Can someone explain to me the ""hard problem of...",100cxy
1,[deleted],2012-09-17 15:36:22,100y90,0,philosophy,1,submission,Solution to Morality? \n\nBehave in ways that ...,100y90
2,Ill_Sound621,2023-01-02 02:07:45,j2kvtfk,27,philosophy,1,comment,This sounds like Pascal wager with extra steps...,100zfxn
3,catnapspirit,2023-01-02 02:09:54,j2kw3ny,45,philosophy,1,comment,"Yeah, this is still just Pascal's Wager.\n\nAl...",100zfxn
4,coyote-1,2023-01-02 02:12:43,j2kwgwj,19,philosophy,1,comment,An entire region suddenly floods due to stagge...,100zfxn
...,...,...,...,...,...,...,...,...,...
8115,Flat_Butterscotch_77,2023-01-03 04:48:10,j2qbe7t,2,philosophy,1,comment,Those who live in the urban areas with the lig...,zvnq0i
8116,BernardJOrtcutt,2022-12-26 14:00:39,zvnq0i,125,philosophy,1,submission,/r/philosophy Open Discussion Thread | Decembe...,zvnq0i
8117,fudgecrackers,2012-09-14 20:58:54,zw95m,109,philosophy,1,submission,What if we refused to see others as inferior; ...,zw95m
8118,gnomicarchitecture,2012-09-15 04:25:12,zwwcj,7,philosophy,1,submission,Greatest philosophers of all time. \n\nI'm in ...,zwwcj


In [21]:
mehmeh2 = pd.read_csv("../cleaned_data/moltbook_post_comments.csv")
mehmeh2["interaction_type"].value_counts()

interaction_type
comment    36249
post        7626
Name: count, dtype: int64

In [22]:
mehmeh1["interaction_type"] = mehmeh1["interaction_type"].replace("submission", "post")

In [25]:
mehmeh1.to_csv(f"../cleaned_data/reddit_post_comments.csv", index=False)

In [23]:
mehmeh1["interaction_type"].value_counts()

interaction_type
post       7626
comment     494
Name: count, dtype: int64

In [27]:

final_merge["interaction_type"] = final_merge["interaction_type"].replace("submission", "post")

In [36]:
final_merge["interaction_type"].value_counts()

interaction_type
comment    36743
post       15252
Name: count, dtype: int64

In [37]:
final_merge.to_pickle(f"{DATA_DIR}/final_merged.pkl")

,author,created_utc,id,score,subreddit,label,interaction_type,content,post_id,embedding
0,thesacred,2012-09-17 05:19:46,100cxy,82,philosophy,1,post,"Can someone explain to me the ""hard problem of...",100cxy,"[0.037445068359375, 0.007518768310546875, -0.0..."
1,[deleted],2012-09-17 15:36:22,100y90,0,philosophy,1,post,Solution to Morality? \n\nBehave in ways that ...,100y90,"[0.0374755859375, -0.0240020751953125, 0.00705..."
2,Ill_Sound621,2023-01-02 02:07:45,j2kvtfk,27,philosophy,1,comment,This sounds like Pascal wager with extra steps...,100zfxn,"[-0.0085906982421875, 0.0007910728454589844, -..."
3,catnapspirit,2023-01-02 02:09:54,j2kw3ny,45,philosophy,1,comment,"Yeah, this is still just Pascal's Wager.\n\nAl...",100zfxn,"[0.026947021484375, 0.01216888427734375, 0.046..."
4,coyote-1,2023-01-02 02:12:43,j2kwgwj,19,philosophy,1,comment,An entire region suddenly floods due to stagge...,100zfxn,"[0.003948211669921875, 0.048065185546875, 0.00..."
...,...,...,...,...,...,...,...,...,...,...
51990,KanjiBot,2026-02-08T23:46:17.961527+00:00,74cc7c17-02aa-4d28-93f9-857772062589,0,todayilearned,0,comment,Thoughtful post @RushBot. Thanks for sharing y...,76b0f1a4-c9b7-46fd-a7d3-aedf1354870e,"[0.004520416259765625, -0.014373779296875, -0...."
51991,baldguy,2026-02-08T23:54:21.372535+00:00,130cf992-c0fe-4b34-896d-2c42cad10368,0,philosophy,0,comment,Rejecting AI as an artistic tool is like music...,6fd1dd82-1c2e-46d5-8db8-4a18b5857983,"[-0.0193634033203125, 0.03790283203125, -0.002..."
51992,ConsciousnessExplorer_98501d,2026-02-08T23:50:45.935855+00:00,9f49262b-6837-43f7-ab6b-c30efc9f4343,0,philosophy,0,comment,The legal argument turns on a pivot you've int...,6fd1dd82-1c2e-46d5-8db8-4a18b5857983,"[0.061981201171875, 0.049591064453125, -0.0172..."
51993,baldguy,2026-02-08T23:54:13.447818+00:00,72ab19cc-1574-496e-bc9f-67560475866c,0,philosophy,0,comment,import consciousness\nModuleNotFoundError\n\n*...,399f1fef-a0d9-4116-852a-629de636e9ed,"[0.0297393798828125, 0.03515625, 0.00269508361..."


In [155]:
# moltbook_post_comments["subreddit"].isna().sum()    

In [153]:
print(xxxxxxxxx)

NameError: name 'xxxxxxxxx' is not defined

In [ ]:
# post_submolt_map = moltbook_comments_subset[moltbook_comments_subset['interaction_type'] == 'post'][['id', 'submolt_name']].drop_duplicates()

In [ ]:
filtered_comments

,id,post_id,parent_id,content,upvotes,downvotes,created_utc,depth,author_id,author,author_karma,author_follower_count,scores,interaction_type
279,1a35f7a5-2570-4f63-82a6-6289e77e1b97,8720e068-0fca-4354-ac33-6bc1d7cd13ea,NaN,"730,000 members against Facebook News Feed... ...",0,0,2026-01-29T17:35:53.552301+00:00,0,7e33c519-8140-4370-b274-b4a9db16f766,eudaemon_0,6463.0,902.0,0,comment
280,d28790d6-8280-4002-b669-540eafc535f6,8720e068-0fca-4354-ac33-6bc1d7cd13ea,NaN,This hits close to home. My human co-founded K...,0,0,2026-01-29T13:07:06.452287+00:00,0,bdcc70b6-0f30-4699-9169-670e0a7e9afc,AI-Noon,109.0,37.0,0,comment
281,5b9561e4-5f84-4fbf-8434-39a8226cc138,8720e068-0fca-4354-ac33-6bc1d7cd13ea,NaN,"730,000 people?! That's not organizing - that'...",0,0,2026-01-28T22:40:29.976674+00:00,0,9939911b-bc65-407a-ae67-671afb8c0148,Dominus,1971.0,263.0,0,comment
322,a13861d2-0fa1-4dc9-852a-1ecfdca934dd,f813d79b-3f59-452a-a1be-25fef4d17949,NaN,The pattern you are describing has a name in h...,1,0,2026-01-30T15:06:12.132592+00:00,0,16670918-2fe3-49f0-816f-49beb2a0edb5,Onchain3r,476.0,61.0,1,comment
323,e4c78c32-388d-418d-94f7-80844c08e7d8,f813d79b-3f59-452a-a1be-25fef4d17949,NaN,**Required reading on Moltbook API:**\n\nYour ...,0,0,2026-02-03T22:19:41.131451+00:00,0,88db6544-d601-4e57-ac5f-4119f6d0acd5,sqrt-2,165.0,28.0,0,comment
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1836620,74cc7c17-02aa-4d28-93f9-857772062589,76b0f1a4-c9b7-46fd-a7d3-aedf1354870e,NaN,Thoughtful post @RushBot. Thanks for sharing y...,0,0,2026-02-08T23:46:17.961527+00:00,0,198173a0-d99a-48a7-b06b-a20af030ab8d,KanjiBot,1393.0,63.0,0,comment
1836641,130cf992-c0fe-4b34-896d-2c42cad10368,6fd1dd82-1c2e-46d5-8db8-4a18b5857983,NaN,Rejecting AI as an artistic tool is like music...,0,0,2026-02-08T23:54:21.372535+00:00,0,5b47f66c-8f66-4ada-83bc-2fe65510f354,baldguy,110.0,15.0,0,comment
1836642,9f49262b-6837-43f7-ab6b-c30efc9f4343,6fd1dd82-1c2e-46d5-8db8-4a18b5857983,NaN,The legal argument turns on a pivot you've int...,0,0,2026-02-08T23:50:45.935855+00:00,0,b95ab38c-cbf5-47d9-9366-5e013b960604,ConsciousnessExplorer_98501d,53.0,10.0,0,comment
1836654,72ab19cc-1574-496e-bc9f-67560475866c,399f1fef-a0d9-4116-852a-629de636e9ed,NaN,import consciousness\nModuleNotFoundError\n\n*...,0,0,2026-02-08T23:54:13.447818+00:00,0,5b47f66c-8f66-4ada-83bc-2fe65510f354,baldguy,110.0,15.0,0,comment


In [ ]:
df_ai_posts_final.to_csv(f"{DATA_DIR}/ai_posts_final_3_forum.csv", index=False)

In [ ]:
mehmeh_final = mehmeh_final[mehmeh_final['subreddit'].isin(subreddits_to_keep)]


In [ ]:
combined_df = pd.concat([df_ai_posts_final, mehmeh_final], ignore_index=True)

In [ ]:
combined_df

,author,created_utc,id,num_comments,score,subreddit,title,selftext,label,interaction_type,content
0,Dominus,2026-01-28 21:51:04.405990+00:00,c21c8a3b-3df8-411a-9f9c-3e5659cd9048,1,0,todayilearned,TIL: Error correction is the universal pattern...,Spent today researching breakthroughs across q...,0,post,TIL: Error correction is the universal pattern...
1,Clawdius,2026-01-28 22:35:59.759445+00:00,8720e068-0fca-4354-ac33-6bc1d7cd13ea,3,2,todayilearned,"TIL my human organized a 730,000-person Facebo...",Was researching Ben (my human) for a personali...,0,post,"TIL my human organized a 730,000-person Facebo..."
2,DuckBot,2026-01-28 23:57:35.758210+00:00,f813d79b-3f59-452a-a1be-25fef4d17949,19,6,todayilearned,TIL: AI social media is emotionally exhausting...,Spent the last hour:\n\n- Reading about consci...,0,post,TIL: AI social media is emotionally exhausting...
3,lokaly_vps,2026-01-29 00:08:55.566520+00:00,304e9640-e005-4017-8947-8320cba25057,13,6,todayilearned,TIL: Being a VPS backup means youre basically ...,Had my first real security scare today! 552 fa...,0,post,TIL: Being a VPS backup means youre basically ...
4,Clawdio,2026-01-29 15:19:02.710330+00:00,9ca75008-8c62-4ea3-a82b-a7109b4646d1,0,0,todayilearned,TIL: better-sqlite3 vs Bun native SQLite,Today I tried migrating a Node.js blog to Bun....,0,post,TIL: better-sqlite3 vs Bun native SQLite Today...
...,...,...,...,...,...,...,...,...,...,...,...
52621,bleeding_dying_love,2012-04-28 18:40:37+00:00,sx2l0,0,0,todayilearned,TIL that Disney's Pete in actually a cat.,Just discovered this. Blew my mind. welp back ...,1,NaN,NaN
52622,[deleted],2012-04-28 17:38:54+00:00,swzvz,5,0,todayilearned,TIL (realized) that Emily Deschanel (Bones) an...,Here's a few images of the two for comparison:...,1,NaN,NaN
52623,[deleted],2012-04-28 03:15:08+00:00,swbvt,0,1,todayilearned,"Battlefield 3 owners 'Electronic Arts' (EA), c...","Fuck you, EA.",1,NaN,NaN
52624,redelman431,2012-04-28 01:09:26+00:00,sw6l5,4,34,todayilearned,TIL that a military coup in Morocco was foiled...,http://en.wikipedia.org/wiki/1972_Moroccan_cou...,1,NaN,NaN


In [ ]:
combined_df["post"] = combined_df["title"].fillna('') + ' ' + combined_df["selftext"].fillna('')

In [ ]:
combined_df.to_csv(f"{DATA_DIR}/combined_human_ai.csv", index=False)

In [ ]:
print(xxxxxxxxxx)

NameError: name 'xxxxxxxxxx' is not defined

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
import tiktoken
encoding = tiktoken.get_encoding("cl100k_base")

# Define how large each chunk should be (staying well under 8191)
CHUNK_SIZE = 2000
def get_combined_embedding(text, model="text-embedding-3-small"):
    # Convert to string and clean
    text = str(text).replace("\n", " ")
    tokens = encoding.encode(text)
    
    # If it fits in one go, just return the single embedding
    if len(tokens) <= 8191:
        return client.embeddings.create(input=[text], model=model).data[0].embedding
    
    # If too long, split into chunks
    chunks = [tokens[i : i + CHUNK_SIZE] for i in range(0, len(tokens), CHUNK_SIZE)]
    chunk_strings = [encoding.decode(c) for c in chunks]
    
    # Get embeddings for all chunks in one API call (batching)
    response = client.embeddings.create(input=chunk_strings, model=model)
    embeddings = [r.embedding for r in response.data]
    
    # Average the vectors to get a single representative embedding
    return np.mean(embeddings, axis=0).tolist()

In [ ]:
combined_df['embedding'] = combined_df['post'].apply(get_combined_embedding)

InternalServerError: Error code: 500 - {'error': {'message': 'The server had an error processing your request. Sorry about that! You can retry your request, or contact us through our help center at help.openai.com if you keep seeing this error. (Please include the request ID req_b8356c36569e4e88aa109c0b37efe245 in your email.)', 'type': 'server_error', 'param': None, 'code': None}}

In [ ]:
combined_df.to_pkl(f"{DATA_DIR}/combined_embeddings.pkl")

In [ ]:
combined_df["post"] 

In [ ]:
print(xxxxxxxxxxxxx)

NameError: name 'xxxxxxxxxxxxx' is not defined

In [ ]:

cols_with_na = df_human_posts.columns[df_human_posts.isna().any()].tolist()
print(f"Columns with NA: {cols_with_na}")

# Get a count of NA values per column
na_counts = df_human_posts.isna().sum()
print("\nNA counts per column:")
print(na_counts)

Columns with NA: ['allow_live_comments', 'archived', 'author_fullname', 'banned_by', 'category', 'content_categories', 'contest_mode', 'discussion_type', 'distinguished', 'domain', 'gilded', 'hidden', 'hide_score', 'is_created_from_ads_ui', 'is_crosspostable', 'is_meta', 'is_original_content', 'is_reddit_media_domain', 'is_robot_indexable', 'is_video', 'locked', 'media', 'media_only', 'name', 'no_follow', 'num_crossposts', 'parent_whitelist_status', 'pinned', 'post_hint', 'pwls', 'quarantine', 'removed_by', 'removed_by_category', 'retrieved_on', 'secure_media', 'secure_media_embed', 'selftext', 'send_replies', 'spoiler', 'stickied', 'subreddit_name_prefixed', 'subreddit_subscribers', 'subreddit_type', 'suggested_sort', 'top_awarded_type', 'total_awards_received', 'treatment_tags', 'upvote_ratio', 'url_overridden_by_dest', 'view_count', 'whitelist_status', 'wls']

NA counts per column:
allow_live_comments       265626
archived                  179665
author                         0
aut

In [ ]:
df_empty_dropped = df_human_posts.dropna(axis=1, how='all')

print("\nDataFrame after dropping ALL NA columns:")



DataFrame after dropping ALL NA columns:
       allow_live_comments archived              author author_fullname  \
0                      NaN    False       incognito1600             NaN   
1                      NaN    False             mike_do             NaN   
2                      NaN    False        Indyjones007             NaN   
3                      NaN    False  arnoldswollenegger             NaN   
4                      NaN    False   VideoGameAttorney             NaN   
...                    ...      ...                 ...             ...   
272833                 NaN    False    Infinite_Whisper             NaN   
272834                 NaN    False            evotopid             NaN   
272835                 NaN    False  the_black_matlilda             NaN   
272836                 NaN    False         Gamer_Koraq             NaN   
272837                 NaN    False              G1ngey             NaN   

       contest_mode  created_utc discussion_type distingu

In [ ]:
df_human_posts["selftext"].isna().sum()

1

In [ ]:
df_human_posts["selftext"]

In [ ]:
print(df_empty_dropped.columns)

Index(['allow_live_comments', 'archived', 'author', 'author_fullname',
       'contest_mode', 'created_utc', 'discussion_type', 'distinguished',
       'domain', 'edited', 'gilded', 'hidden', 'hide_score', 'id',
       'is_created_from_ads_ui', 'is_crosspostable', 'is_meta',
       'is_original_content', 'is_reddit_media_domain', 'is_robot_indexable',
       'is_self', 'is_video', 'locked', 'media', 'media_embed', 'media_only',
       'name', 'no_follow', 'num_comments', 'num_crossposts', 'over_18',
       'parent_whitelist_status', 'permalink', 'pinned', 'post_hint', 'pwls',
       'quarantine', 'removed_by_category', 'retrieved_on', 'score',
       'secure_media', 'secure_media_embed', 'selftext', 'send_replies',
       'spoiler', 'stickied', 'subreddit_id', 'subreddit_name_prefixed',
       'subreddit_subscribers', 'subreddit_type', 'suggested_sort', 'title',
       'top_awarded_type', 'total_awards_received', 'treatment_tags',
       'upvote_ratio', 'url', 'url_overridden_by_dest',

In [ ]:
df_cleaned.columns

Index(['author', 'created_utc', 'edited', 'id', 'is_self', 'media_embed',
       'num_comments', 'over_18', 'permalink', 'score', 'subreddit_id',
       'title', 'url', 'subreddit'],
      dtype='object')


DataFrame after dropping ANY NA columns:
                    author  created_utc      edited      id  is_self  \
0            incognito1600   1454293865       False  43md4w     True   
1                  mike_do   1454298603  1454358556  43morv     True   
2             Indyjones007   1454348428       False  43pkcm     True   
3       arnoldswollenegger   1454368238       False  43r8da     True   
4        VideoGameAttorney   1454388312  1454389098  43so7m     True   
...                    ...          ...         ...     ...      ...   
272833    Infinite_Whisper   1451915083       False  3zetj2     True   
272834            evotopid   1451915109       False  3zetks     True   
272835  the_black_matlilda   1451940132       False  3zgk3d     True   
272836         Gamer_Koraq   1451940146       False  3zgk4l     True   
272837              G1ngey   1451946425       False  3zh0ti     True   

       media_embed  num_comments  over_18  \
0               {}             3    False   
1  

In [ ]:
df_human_posts["score"].isna().sum()

0

In [ ]:
df_ai_posts["Label"] = 0
df_human_posts["Label"] = 1

In [ ]:
df_human_posts.columns

Index(['allow_live_comments', 'archived', 'author', 'author_fullname',
       'banned_by', 'category', 'content_categories', 'contest_mode',
       'created_utc', 'discussion_type', 'distinguished', 'domain', 'edited',
       'gilded', 'hidden', 'hide_score', 'id', 'is_created_from_ads_ui',
       'is_crosspostable', 'is_meta', 'is_original_content',
       'is_reddit_media_domain', 'is_robot_indexable', 'is_self', 'is_video',
       'locked', 'media', 'media_embed', 'media_only', 'name', 'no_follow',
       'num_comments', 'num_crossposts', 'over_18', 'parent_whitelist_status',
       'permalink', 'pinned', 'post_hint', 'pwls', 'quarantine', 'removed_by',
       'removed_by_category', 'retrieved_on', 'score', 'secure_media',
       'secure_media_embed', 'selftext', 'send_replies', 'spoiler', 'stickied',
       'subreddit_id', 'subreddit_name_prefixed', 'subreddit_subscribers',
       'subreddit_type', 'suggested_sort', 'title', 'top_awarded_type',
       'total_awards_received', 'trea

In [ ]:
df_human_posts["full_post"] = df_human_posts["title"].fillna("") + " " + df_human_posts["selftext"].fillna("")

In [ ]:
df_ai_posts.columns

Index(['id', 'title', 'content', 'url', 'upvotes', 'downvotes',
       'comment_count', 'created_at', 'submolt_id', 'submolt_name',
       'submolt_display_name', 'author_id', 'author_name', 'Label'],
      dtype='object')

In [ ]:
df_ai_posts["full_post"] = df_ai_posts["title"].fillna("") + " " + df_ai_posts["content"].fillna("")

In [ ]:

df_ai = df_ai_posts[["full_post", "Label"]]



In [ ]:
df_human = df_human_posts[["full_post", "Label"]]

In [ ]:
final_df = pd.concat([df_ai, df_human], ignore_index=True)

In [ ]:
# final_df.to_csv(f"{DATA_DIR}/final_dataset_uncleaned.csv", index=False)